In [1]:
import sys
print(sys.executable)
!pip install XlsxWriter


C:\Users\MOUNIKA PALLI\anaconda3\python.exe


In [12]:
import pandas as pd
import numpy as np
from datetime import datetime
import calendar

# --- File paths ---
input_sales_path = r"C:\Users\MOUNIKA PALLI\Dropbox\EBO FOLDER\EBO SALES FOLDER\EBO SALES DATA.xlsx"
sku_master_path = r"C:\Users\MOUNIKA PALLI\Dropbox\EBO FOLDER\MASTERS\SKU MASTER.xlsx"
exclude_styles_path = r"C:\Users\MOUNIKA PALLI\Dropbox\EBO FOLDER\DT Exc Styles.xlsx"
output_excel_path = "SKU_Level_MRR_Without_Store.xlsx"

# --- Freebie SKUs to exclude ---
FREEBIE_SKUS = [
    'MABOATPB3990', 'MAAIRPODS4999', 'MAPOWBANK3999',
    'MAAIRPODS3599', 'MATROLLEY9999', 'UAGY01BLK699', 'MAKRAFTBAGBIG'
]

def exclude_freebies(df, sku_column='SKU'):
    """Return df excluding known freebie SKUs."""
    return df[~df[sku_column].isin(FREEBIE_SKUS)].copy()

def get_last_n_months_ranges(max_date, n=4):
    """Return dictionary of last n months ranges {label:(start,end)}."""
    months = {}
    current = max_date.replace(day=1)
    for i in range(n):
        month_label = current.strftime("%b").upper()  # e.g., JUL
        start = current
        end = current.replace(day=calendar.monthrange(current.year, current.month)[1])
        months[month_label] = (start, end)
        # Move one month back
        prev_month = current.month - 1 or 12
        prev_year = current.year if current.month > 1 else current.year - 1
        current = current.replace(year=prev_year, month=prev_month, day=1)
    return dict(reversed(months.items()))  # Keep chronological order

def calculate_mrr_all_stores(sales_df):
    """Calculate monthly MRRs and final MRR calculations."""
    df = sales_df.copy()

    # Ensure required columns exist
    req_cols = ['BILL_DATE', 'BILL_QUANTITY', 'SKU']
    for col in req_cols:
        if col not in df.columns:
            raise ValueError(f"Input sales file must contain '{col}' column")

    df['DATE'] = pd.to_datetime(df['BILL_DATE'], errors='coerce')
    df.rename(columns={'BILL_QUANTITY': 'QTY'}, inplace=True)

    # Get last 4 months dynamically
    max_date = df['DATE'].max()
    month_defs = get_last_n_months_ranges(max_date, n=4)

    monthly_results = []
    for m, (start, end) in month_defs.items():
        month_sales = exclude_freebies(df[(df['DATE'] >= start) & (df['DATE'] <= end)])
        grouped = month_sales.groupby('SKU', as_index=False)['QTY'].sum().rename(columns={'QTY': f'MRR_{m}'})
        monthly_results.append(grouped)

    # Merge all months
    merged = monthly_results[0]
    for gr in monthly_results[1:]:
        merged = pd.merge(merged, gr, on='SKU', how='outer')

    merged.fillna(0, inplace=True)

    # Ensure numeric
    for col in merged.columns:
        if col.startswith("MRR_"):
            merged[col] = pd.to_numeric(merged[col], errors='coerce').fillna(0)

    # Best MRR (based on last 3 months except the earliest)
    last_months = list(month_defs.keys())[-3:]
    merged['BEST_MRR'] = merged[[f"MRR_{m}" for m in last_months]].max(axis=1)

    # New MRR calc (same logic as your version)
    merged['NEW_MRR'] = (merged['BEST_MRR'] / 9.0) * 2.0
    latest_month = list(month_defs.keys())[-1]
    merged[f'MRR_{latest_month}_FINAL'] = ((merged['BEST_MRR'] + merged['NEW_MRR']) * 1.2).round(0)

    # Requirement
    merged['REQ'] = (merged['BEST_MRR'] * 4 * 4 * 1.3).round(0)

    return merged, month_defs

# ------------------- MAIN SCRIPT -------------------

# Load sales data
sales_df = pd.read_excel(input_sales_path)

# Calculate MRR
final_mrr_df, month_defs = calculate_mrr_all_stores(sales_df)

# Load SKU Master & clean duplicates
sku_master_df = pd.read_excel(sku_master_path)
sku_master_df.columns = sku_master_df.columns.str.strip().str.upper()
sku_master_df = sku_master_df.loc[:, ~sku_master_df.columns.duplicated()]

# Ensure all needed columns exist in SKU master
columns_to_merge = ['SKU', 'STYLE', 'COLOUR', 'SIZE', 'SIZE MAP', 'SIZE NUM', 'CODE']
for c in columns_to_merge:
    if c not in sku_master_df.columns:
        sku_master_df[c] = pd.NA

# Merge SKU info
final_mrr_df.columns = final_mrr_df.columns.str.strip().str.upper()
final_mrr_df = final_mrr_df.merge(sku_master_df[columns_to_merge], on='SKU', how='left')

# Derived columns
final_mrr_df['STYLE*CODE'] = final_mrr_df.get('STYLE', "").astype(str) + "*" + final_mrr_df.get('CODE', "").astype(str)
final_mrr_df['STYLE COLOUR'] = final_mrr_df.get('STYLE', "").astype(str) + " " + final_mrr_df.get('COLOUR', "").astype(str)

# 3-month avg MRR
last_3 = list(month_defs.keys())[-3:]
for m in last_3:
    col = f"MRR_{m}"
    if col not in final_mrr_df.columns:
        final_mrr_df[col] = 0
final_mrr_df['AVG_MRR_3M'] = final_mrr_df[[f"MRR_{m}" for m in last_3]].mean(axis=1).round(2)

# DRR per month (vectorized)
for m in last_3:
    month_col = f"MRR_{m}"
    drr_col = f"DRR_{m}"
    final_mrr_df[drr_col] = np.where(final_mrr_df[month_col] > 0,
                                     final_mrr_df['REQ'] / final_mrr_df[month_col],
                                     0)

# Best DRR
drr_cols = [f"DRR_{m}" for m in last_3]
final_mrr_df['BEST_DRR'] = final_mrr_df[drr_cols].max(axis=1).round(2)

# Ordered columns dynamically
ordered_columns = (
    ['SKU', 'STYLE', 'CODE', 'STYLE*CODE', 'STYLE COLOUR', 'COLOUR', 'SIZE', 'SIZE MAP', 'SIZE NUM'] +
    [f"MRR_{m}" for m in month_defs.keys()] +
    [f"MRR_{list(month_defs.keys())[-1]}_FINAL", 'AVG_MRR_3M', 'BEST_MRR', 'BEST_DRR', 'REQ']
)
for c in ordered_columns:
    if c not in final_mrr_df.columns:
        final_mrr_df[c] = pd.NA
final_mrr_df = final_mrr_df[ordered_columns]

# Pivots
pivot = pd.pivot_table(
    final_mrr_df,
    index=['STYLE', 'CODE', 'STYLE*CODE'],
    columns='SIZE MAP',
    values='REQ',
    aggfunc='sum',
    fill_value=0,
    margins=True,
    margins_name='Grand Total'
)

pivot_style_colour = pd.pivot_table(
    final_mrr_df,
    index=['STYLE', 'STYLE COLOUR'],
    values=['REQ', 'AVG_MRR_3M', 'BEST_MRR', 'BEST_DRR'],
    aggfunc={'REQ':'sum', 'AVG_MRR_3M':'mean', 'BEST_MRR':'mean', 'BEST_DRR':'mean'},
    fill_value=0,
    margins=True,
    margins_name='Grand Total'
)

# Exclude styles (safe)
try:
    exclude_styles_df = pd.read_excel(exclude_styles_path)
    exclude_styles_df.columns = exclude_styles_df.columns.str.strip().str.upper()
    if 'STYLE' not in exclude_styles_df.columns:
        exclude_styles_df['STYLE'] = pd.NA
    exclude_styles_list = exclude_styles_df['STYLE'].astype(str).str.strip().str.upper().tolist()
except Exception:
    exclude_styles_list = []

filtered_mrr_df = final_mrr_df[
    ~final_mrr_df['STYLE'].astype(str).str.strip().str.upper().isin(exclude_styles_list)
].copy()

pivot_filtered = pd.pivot_table(
    filtered_mrr_df,
    index=['STYLE', 'CODE', 'STYLE*CODE'],
    columns='SIZE MAP',
    values='REQ',
    aggfunc='sum',
    fill_value=0,
    margins=True,
    margins_name='Grand Total'
)

# Save all sheets with formatting
with pd.ExcelWriter(output_excel_path, engine='xlsxwriter') as writer:
    final_mrr_df.to_excel(writer, sheet_name='SKU_MRR_Detail', index=False)
    pivot.to_excel(writer, sheet_name='Pivot_Style_Code', merge_cells=False)
    pivot_style_colour.to_excel(writer, sheet_name='Pivot_Style_Colour', merge_cells=False)
    filtered_mrr_df.to_excel(writer, sheet_name='Filtered_MRR_No_ExcStyles', index=False)
    pivot_filtered.to_excel(writer, sheet_name='Pivot_Filtered_No_ExcStyles', merge_cells=False)

    # Apply Indian comma formatting
    workbook = writer.book
    comma_fmt = workbook.add_format({'num_format': '#,##,##0'})
    for sheet in ['SKU_MRR_Detail', 'Filtered_MRR_No_ExcStyles']:
        ws = writer.sheets[sheet]
        ws.set_column('J:Z', 15, comma_fmt)

print(f"✅ Output saved to: {output_excel_path}")


✅ Output saved to: SKU_Level_MRR_Without_Store.xlsx
